```markdown
# ComfyUI Photorealistic Indian Influencer Setup
1. Run the Environment Setup.
2. Download the required models.
3. Enter your ngrok token and launch.
```

In [52]:
#@title 1. Environment Setup
import os

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
else:
    WORKSPACE = "/content/ComfyUI"

if not os.path.exists(WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}

%cd {WORKSPACE}
!pip install pyngrok xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ComfyUI
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121


In [65]:
#@title 2. Verify Models & Cleanup
import os

%cd {WORKSPACE}
os.makedirs('./models/checkpoints/', exist_ok=True)
os.makedirs('./models/upscale_models/', exist_ok=True)

ckpt_path = './models/checkpoints/realvisxlV50_v50Bakedvae.safetensors'

# Automatic cleanup of corrupt placeholders
if os.path.exists(ckpt_path):
    size = os.path.getsize(ckpt_path)
    if size < 1000000: # Less than 1MB is definitely a failed download
        print(f"☑️ Removing corrupt file placeholder ({size} bytes)...")
        os.remove(ckpt_path)
        print("✅ Done. Please upload the full 6.5GB model to /ComfyUI/models/checkpoints/ now.")
    else:
        print(f"✅ RealVisXL V5.0 detected ({size / (1024**3):.2f} GB).")
else:
    print("⌛ RealVisXL not found. Waiting for manual upload to: ComfyUI/models/checkpoints/")

# 4x-UltraSharp
print("\nChecking 4x-UltraSharp...")
!wget -c -L "https://huggingface.co/lokCX/4x-UltraSharp/resolve/main/4x-UltraSharp.pth" -P ./models/upscale_models/

/content/drive/MyDrive/ComfyUI
☑️ Removing corrupt file placeholder (0 bytes)...
✅ Done. Please upload the full 6.5GB model to /ComfyUI/models/checkpoints/ now.

Checking 4x-UltraSharp...
--2026-07-31 09:47:36--  https://huggingface.co/lokCX/4x-UltraSharp/resolve/main/4x-UltraSharp.pth
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.23, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth [following]
--2026-07-31 09:47:36--  https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/6430096543a53c86b3fcb2a0/080be486975ca8916a1a6fda9e763332b218dbc306ca993f2a029595919e72be?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+file

In [ ]:
#@title 3. Launch ComfyUI via Ngrok (with Real-time Logs)
from pyngrok import ngrok
import threading
import subprocess
import sys
import os

# Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "3HG2NXigok7WNH2YMXhlymV3M1j_2YP3SwKDSroFHjKj87L3B" #@param {type:"string"}
PORT = 8188

def start_ngrok(port):
    if not NGROK_AUTH_TOKEN:
        print("❌ ERROR: Please enter your ngrok token!")
        return
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok.kill()
    url = ngrok.connect(port, "http").public_url
    print(f"\n[Ready] URL: {url}\n")

def run_comfy():
    os.chdir(WORKSPACE)
    # Bound to 0.0.0.0 for ngrok compatibility and explicitly set cuda-device 0
    process = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(PORT), "--enable-cors-header", "--force-fp16", "--highvram", "--cuda-device", "0"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end="")

# Start tunnel
if NGROK_AUTH_TOKEN:
    threading.Thread(target=start_ngrok, args=(PORT,), daemon=True).start()

# Start ComfyUI with logging
run_comfy()


[Ready] URL: https://silk-greeter-craziness.ngrok-free.dev

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] Set cuda device to: 0


[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'int8_linear', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'rms_adaln', 'rms_rope', 'rms_rope1', 'rms_rope1_', 'rms_rope_', 'rms_rope_split_half', 'rms_rope_split_half1', 'rms_rope_split_half1_', 'rms_rope_split_half_']}
[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason'

[INFO] Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO] ComfyUI version: 0.29.0
[INFO] comfy-aimdo version: 0.4.10
[INFO] comfy-kitchen version: 0.2.25
[INFO] comfyui-frontend-package version: 1.47.11
[INFO] comfyui-workflow-templates version: 0.11.20
[INFO] comfyui-embedded-docs version: 0.5.9
[INFO] comfy-kitchen version: 0.2.25
[INFO] comfy-aimdo version: 0.4.10
[INFO] [Prompt Server] web root: /usr/local/lib/python3.12/dist-packages/comfyui_frontend_package/static
[INFO] Asset seeder disabled


[INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'
[INFO] 
Import times for custom nodes:
[INFO]    0.0 seconds: /content/drive/MyDrive/ComfyUI/custom_nodes/websocket_image_save.py
[INFO] 
[INFO] Context impl SQLiteImpl.
[INFO] Will assume non-transactional DDL.
[INFO] Using RAM pressure cache.
[INFO] Starting server

[INFO] To see the GUI go to: http://0.0.0.0:8188
[INFO] got prompt
[INFO] model weight dtype torch.float16, manual cast: None
[INFO] model_type EPS
[INFO] Using xformers attention in VAE
[INFO] Using xformers attention in VAE
[INFO] VAE load device: cuda:0, offload device: cpu, dtype: torch.float32
[INFO] Requested to load SDXLClipModel
[INFO] loaded completely;  1560.80 MB loaded, full load: True
[INFO] CLIP/text encoder model load device: cuda:0, offload device: cpu, current: cuda:0, dtype: torch.float16
[INFO] loaded diffusion model directly to GPU
[INFO] Requested to load SDXL
[INFO] loaded completely;  4897.05 MB loaded, full load: True

 